# 5.8 · 梯度提升分类 / Gradient Boosting (GBDT) Classifier

> **课程定位 / Where this fits**
> 第 8 课，**Part 5 · 监督学习：分类**。
> Lesson 8, **Part 5 · Supervised Classification**.
>
> 5.7 的随机森林是 **bagging**（并行、独立、降方差）。GBDT 是 **boosting**（串行、每棵树纠正前面的错、降偏差）。4.13 讲过回归版的梯度提升，这一课是分类版，并把它当作 5.9–5.11 三巨头（XGBoost/LightGBM/CatBoost）的**数学地基**。
> Random Forest (5.7) is **bagging** (parallel, independent, variance reduction). GBDT is **boosting** (sequential, each tree fixes the previous error, bias reduction). 4.13 covered the regression version; this is classification, and the **mathematical foundation** for the big three (5.9–5.11).

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $F(\mathbf{x})$ —— 当前加法模型的输出（在 log-odds 空间）/ current additive model output (in log-odds space)
> - $h_m$ —— 第 $m$ 棵树 / the $m$-th tree
> - $\nu$ (learning rate) —— 学习率/收缩系数 / shrinkage
> - $r_{im}$ —— 伪残差（损失的负梯度）/ pseudo-residual (negative gradient of loss)
> - $p_i=\sigma(F(\mathbf{x}_i))$ —— 预测概率 / predicted probability

> 💡 **面试相关 / Interview-relevant**
> - "boosting 与 bagging 的本质区别"（★★★★★）
> - "梯度提升的'梯度'是对什么求导"（★★★★★，对预测值/函数）
> - "GBDT 分类如何工作（拟合 log-odds 残差）"（★★★★）
> - "学习率(shrinkage)的作用"（★★★★★）
> - "GBDT 为什么用浅树 / 怎么防过拟合"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 boosting 思想及其与 bagging 的对立。
   Understand boosting and its contrast with bagging.
2. 理解**函数空间的梯度下降**：每棵树拟合损失的负梯度（伪残差）。
   Understand functional gradient descent: each tree fits the negative gradient (pseudo-residual).
3. 看清分类版：在 **log-odds** 空间加法建模，伪残差 = $(y-p)$。
   See the classification version: additive modeling in log-odds space, pseudo-residual $=(y-p)$.
4. 理解学习率 × 树数的权衡 + 早停。
   Understand the learning-rate × tree-count trade-off and early stopping.
5. 对照 sklearn 的 `GradientBoosting` / `HistGradientBoosting`。
   Compare sklearn's `GradientBoosting` / `HistGradientBoosting`.

## 目录 / TOC
1. [先建直觉：一棵接一棵补错](#1)
2. [boosting vs bagging ⭐](#2)
3. [函数梯度下降 ⭐](#3)
4. [💰 数据：合成 Adult Income](#4)
5. [从零：二分类 GBDT ⭐](#5)
6. [学习率 × 树数 + 早停 ⭐](#6)
7. [sklearn + HistGBDT](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉：一棵接一棵补错 / Intuition First

随机森林是"很多专家同时投票"。boosting 换了个思路：**一个接一个地请专家，每个新专家专门去补前面所有人犯的错。**
Random Forest is "many experts voting at once". Boosting takes a different tack: **bring in experts one at a time, each new one specializing in fixing the mistakes the others made.**

具体做法：先用一个很弱的模型（一棵浅树）做个粗预测，看它在哪些样本上错得多；再训练下一棵树，**专门去拟合这些"还没解释好的部分"**；把它加进来；如此反复。一堆弱模型叠加，最终变成一个很强的模型。
Concretely: start with a weak model (a shallow tree) for a rough prediction, see where it errs most; train the next tree to **specifically fit the "not-yet-explained" part**; add it in; repeat. Many weak models stacked become a strong one.

下面要把"还没解释好的部分"精确化——它就是损失函数的**负梯度**（伪残差），所以这个方法叫"梯度"提升。
Below we make "not-yet-explained part" precise — it's the **negative gradient** of the loss (the pseudo-residual), hence "gradient" boosting.


<a id="2"></a>
## 2. boosting vs bagging ⭐ / Boosting vs Bagging

| | Bagging (RF, 5.7) | Boosting (GBDT) |
|---|---|---|
| 树的关系 trees | **独立并行** independent, parallel | **串行依赖**，后树修前树的错 sequential |
| 基学习器 base | 深树（低偏差高方差）deep | **浅树**（高偏差低方差，弱学习器）shallow |
| 主攻 target | 降**方差** variance | 降**偏差** bias |
| 加树过多 more trees | 不过拟合 no overfit | **会**过拟合 can overfit |

**boosting 直觉**：先用一个弱模型，看它哪里错了，再训一个新模型**专门补这个错**，加权叠加，反复。无数弱学习器叠成强模型。
**Boosting intuition:** weak model → see its errors → train a new model to **fix exactly those** → add it weighted → repeat. Countless weak learners compound into a strong model.


<a id="3"></a>
## 3. 函数空间的梯度下降 ⭐ / Functional Gradient Descent

GBDT 把模型写成加法形式 $F_M(\mathbf{x})=\sum_{m=1}^M \nu\, h_m(\mathbf{x})$，一棵棵地往上加树。第 $m$ 步，我们想减小损失 $L(y, F)$。
GBDT writes the model additively, $F_M(\mathbf{x})=\sum_{m=1}^M \nu\, h_m(\mathbf{x})$, adding trees one by one. At step $m$ we want to decrease the loss $L(y, F)$.

**关键洞察**：把当前的预测值 $F(\mathbf{x})$ 本身当作"要优化的变量"，对它求梯度。损失下降最快的方向是**负梯度**：
**Key insight:** treat the current prediction $F(\mathbf{x})$ itself as the "variable to optimize" and take the gradient. The steepest-descent direction is the **negative gradient**:

$$r_{im} = -\left[\frac{\partial L(y_i, F(\mathbf{x}_i))}{\partial F(\mathbf{x}_i)}\right]_{F=F_{m-1}}$$

这个量叫**伪残差(pseudo-residual)**。然后训练一棵树 $h_m$ 去**拟合这些伪残差**，再以学习率 $\nu$ 加进模型。
This quantity is the **pseudo-residual**. We then train a tree $h_m$ to **fit those pseudo-residuals** and add it in with learning rate $\nu$.

- **回归 + 平方损失**(4.13)：伪残差 $= y_i - F(\mathbf{x}_i)$，就是**普通残差**（真实减预测）。
  Regression + squared loss: the pseudo-residual is $y_i - F(\mathbf{x}_i)$, the **ordinary residual**.
- **分类 + 对数损失**：$F$ 是 **log-odds**，伪残差 $= y_i - \sigma(F(\mathbf{x}_i)) = y_i - p_i$ —— 又是 $(y-p)$！和 5.1/5.2 同一个量。
  Classification + log loss: $F$ is **log-odds**, and the pseudo-residual is $y_i - \sigma(F(\mathbf{x}_i)) = y_i - p_i$ — once again $(y-p)$, the same quantity as 5.1/5.2.

所以分类 GBDT = **在 log-odds 空间里反复拟合 $(y-p)$ 残差**。
So classification GBDT = **repeatedly fitting the $(y-p)$ residual in log-odds space.**


<a id="4"></a>
## 4. 数据：合成 Adult Income / Synthetic Adult Income

真实 UCI Adult（预测成年人收入是否 >50K）需要下载。这里**内联合成**一个同风格数据集：用年龄、教育年限、每周工时、资本利得等特征，按一个含**非线性 + 交互项**的真实规律生成"高收入"标签。5.8–5.11 都用它，方便横向对比四种 boosting。
The real UCI Adult (predict income >50K) needs downloading. We **inline a same-style synthetic** set: from age, education-years, weekly hours, capital-gain, with a generating rule containing **nonlinearity + interactions**. 5.8–5.11 all use it for a fair cross-comparison of the four boosters.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def make_income(n=8000, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 70, n)
    edu_years = rng.integers(6, 21, n)
    hours = rng.normal(40, 10, n).clip(10, 80)
    capital_gain = (rng.random(n) < 0.15) * rng.exponential(5000, n)   # 85% 的人无资本利得(=0)
    # 真实的 log-odds 规律: 含非线性(年龄平方, 中年收入更高) + 交互(教育×资本利得)
    logit = (-9 + 0.04*age - 0.0004*(age-45)**2 + 0.25*edu_years
             + 0.02*hours + 0.0002*np.sqrt(capital_gain)*edu_years*0.3
             + rng.normal(0, 0.5, n))                                  # 加噪声
    y = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)            # 按概率(sigmoid)采样标签
    X = pd.DataFrame({"age": age, "edu_years": edu_years, "hours": hours,
                      "capital_gain": capital_gain.round(0)})
    return X, y

X, y = make_income()
print(f"合成 Adult Income: {X.shape}, 高收入率 high-income rate {y.mean():.0%}")
print(X.describe().round(1).to_string())

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X.values, y, test_size=0.3, stratify=y, random_state=0)


<a id="5"></a>
## 5. 从零：二分类 GBDT ⭐ / From Scratch

按第 3 节的公式实现：从一个全局 log-odds 常数起步，每一轮算出伪残差 $(y-p)$，**用一棵浅回归树去拟合这些残差**，再乘学习率加进模型。
Following Section 3: start from a global log-odds constant; each round compute the pseudo-residual $(y-p)$, **fit a shallow regression tree to those residuals**, and add it in scaled by the learning rate.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

def sigmoid(z): return 1/(1+np.exp(-np.clip(z, -500, 500)))

class GBDTClassifier:
    def __init__(self, n_trees=100, lr=0.1, max_depth=3):
        self.n_trees, self.lr, self.max_depth = n_trees, lr, max_depth
    def fit(self, X, y):
        # 初始预测 F0 = 全局 log-odds(整体正类比例对应的对数几率), 是最朴素的常数预测
        self.F0 = np.log(y.mean()/(1-y.mean()))
        F = np.full(len(y), self.F0)        # F 是每个样本当前的 log-odds 预测值
        self.trees = []
        for _ in range(self.n_trees):
            p = sigmoid(F)                  # 把 log-odds 转成概率
            residual = y - p                # 伪残差 = 真实 - 预测概率 (对数损失的负梯度)
            # 用回归树去拟合"还没解释好的部分"(伪残差); 注意是回归树, 因为残差是连续值
            tree = DecisionTreeRegressor(max_depth=self.max_depth).fit(X, residual)
            F += self.lr * tree.predict(X)  # 把这棵树的修正量(乘学习率)加进当前预测
            self.trees.append(tree)
        return self
    def decision(self, X):                  # 把所有树的输出累加, 得到最终 log-odds
        F = np.full(len(X), self.F0)
        for t in self.trees: F += self.lr * t.predict(X)
        return F
    def predict_proba(self, X): return sigmoid(self.decision(X))   # log-odds → 概率
    def predict(self, X): return (self.predict_proba(X) > 0.5).astype(int)

gb = GBDTClassifier(n_trees=100, lr=0.1, max_depth=3).fit(X_tr, y_tr)
from sklearn.metrics import accuracy_score, roc_auc_score
print(f"从零 GBDT test 准确率 accuracy: {accuracy_score(y_te, gb.predict(X_te)):.3f}")
print(f"从零 GBDT test AUC:            {roc_auc_score(y_te, gb.predict_proba(X_te)):.3f}")


In [ ]:
# 对照 sklearn / vs sklearn
from sklearn.ensemble import GradientBoostingClassifier
sk = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_tr, y_tr)
print(f"sklearn GBDT test 准确率 accuracy: {sk.score(X_te, y_te):.3f}")
print(f"sklearn GBDT test AUC:            {roc_auc_score(y_te, sk.predict_proba(X_te)[:,1]):.3f}")
print("从零实现与库结果接近 → 验证了'拟合(y-p)伪残差'的核心机制 / confirms the (y-p) mechanism")


<a id="6"></a>
## 6. 学习率 × 树数 + 早停 ⭐ / Learning Rate × Trees & Early Stopping

**学习率 $\nu$（shrinkage）**：每棵树只迈一小步。学习率小 → 需要更多树，但泛化通常更好（类似 SGD 用小步长）。
**Learning rate $\nu$ (shrinkage):** each tree takes only a small step. Small $\nu$ → needs more trees but usually generalizes better (like a small SGD step).

**经验法则**：小 $\nu$（0.01–0.1）+ 多树 + **早停**。这与 RF 相反——GBDT **树太多会过拟合**（它在不断逼近训练集）。
**Rule of thumb:** small $\nu$ (0.01–0.1) + many trees + **early stopping**. Opposite to RF — GBDT **overfits with too many trees** (it keeps chasing the training set).


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for lr in [0.01, 0.1, 0.5]:                          # 试三种学习率
    aucs = []
    for nt in [10, 30, 60, 100, 150, 200]:           # 不同树数
        m = GradientBoostingClassifier(n_estimators=nt, learning_rate=lr, max_depth=3, random_state=0).fit(X_tr, y_tr)
        aucs.append(roc_auc_score(y_te, m.predict_proba(X_te)[:,1]))
    ax.plot([10,30,60,100,150,200], aucs, "o-", label=f"lr={lr}")
ax.set_xlabel("n_estimators 树数"); ax.set_ylabel("test AUC"); ax.legend()
ax.set_title("学习率×树数: 大lr 快但易过拟合; 小lr 需更多树更稳 / small lr needs more trees, generalizes better")
plt.tight_layout(); plt.show()

# 早停: 留出 20% 当验证集, 连续 10 轮验证集不再提升就停 / early stopping on a validation set
es = GradientBoostingClassifier(n_estimators=500, learning_rate=0.1, max_depth=3,
                                validation_fraction=0.2, n_iter_no_change=10, random_state=0).fit(X_tr, y_tr)
print(f"早停: 上限设 500 棵, 实际只用 {es.n_estimators_} 棵就停了(验证集不再提升)")
print(f"早停模型 test AUC: {roc_auc_score(y_te, es.predict_proba(X_te)[:,1]):.3f}")


<a id="7"></a>
## 7. sklearn + HistGBDT / Histogram-based GBDT

sklearn 的 `HistGradientBoostingClassifier` 把连续特征**分箱（直方图）**，大幅加速（借鉴了 LightGBM，见 5.10），在大数据上比经典 `GradientBoosting` 快很多，还原生支持缺失值。
sklearn's `HistGradientBoostingClassifier` **bins continuous features (histograms)** for big speedups (inspired by LightGBM, 5.10), much faster than classic `GradientBoosting` on large data, with native missing-value support.


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
import time
for name, model in [("GradientBoosting", GradientBoostingClassifier(n_estimators=200, random_state=0)),
                    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=200, random_state=0))]:
    t = time.perf_counter(); model.fit(X_tr, y_tr); dt = time.perf_counter()-t
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
    print(f"{name:<22} 训练 train {dt:.2f}s  test AUC {auc:.3f}")
print("注: 仅 8000 行时分箱的固定开销没赚回来, Hist 不一定更快;")
print("    它的优势在'大数据(数十万行+)'时才显著——那时分箱让每次分裂从 O(n) 降到 O(bins)。")
print("HistGBDT 是现代 boosting(5.9-5.11)的思路源头, 大数据优先 / source of modern boosting")


<a id="8"></a>
## 8. 小结 / Summary

```
boosting: 串行, 每棵浅树纠正前面的错, 降偏差 (vs bagging 并行降方差)
函数梯度下降: 树拟合损失的负梯度(伪残差)
  回归+平方损失: 残差 = y - F
  分类+对数损失: 伪残差 = y - σ(F) = y - p (log-odds 空间), 又是 (y-p)
学习率 ν: 小步长更稳, 需更多树; GBDT 树太多会过拟合 → 早停
HistGBDT: 特征分箱加速, 现代 boosting 起点
```

### 💡 面试速查 / Interview cheat-sheet
1. **boosting 串行降偏差**（浅弱树），bagging 并行降方差（深树）。
   Boosting is sequential bias-reduction (shallow weak trees); bagging is parallel variance-reduction (deep trees).
2. **梯度**是对**预测值 $F(\mathbf{x})$** 求导（函数空间），树拟合负梯度=伪残差。
   The gradient is w.r.t. the prediction $F(\mathbf{x})$ (function space); trees fit the negative gradient = pseudo-residual.
3. **分类伪残差 = $y - p$**（在 log-odds 空间加法建模）。
   Classification pseudo-residual = $y - p$ (additive modeling in log-odds space).
4. **学习率小 + 树多 + 早停**；树过多会过拟合（与 RF 不同）。
   Small LR + many trees + early stopping; too many trees overfit (unlike RF).
5. **HistGBDT** 分箱加速，是 LightGBM 思想的体现。
   HistGBDT bins features for speed, embodying LightGBM's idea.

### 下一节 / Next
**5.9 XGBoost**——在 GBDT 上加二阶泰勒近似、正则化叶子权重、列采样等工程与数学改进，Kaggle 表格赛长期霸主。
**5.9 XGBoost** — adds a second-order Taylor objective, leaf-weight regularization, column subsampling and more on top of GBDT; the long-reigning Kaggle tabular champion.
